# 6B · The Forecast Bake-Off — Evaluation Is the Skill
### Financial Analytics — Module 6

Everyone can produce a forecast. The professional skill is **judging forecasts honestly** — and this notebook is that skill, end to end:

1. Three humble forecasters: **naive**, **moving average**, **exponential smoothing**
2. The one unbreakable rule: **the time-respecting split**
3. Scoring: **MAE, RMSE, MAPE** — and what each rewards
4. **Walk-forward** evaluation — how it's really done
5. The punchline about markets that saves you from a thousand bad strategies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

BASE = "data/"
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()

# Monthly closes: 60 points - a classic forecasting canvas
m = px["close"].resample("ME").last().dropna()
print(len(m), "months, from", m.index[0].date(), "to", m.index[-1].date())
m.plot(figsize=(9,3), title="NIFTY 50 - month-end closes", color="#2563EB"); plt.tight_layout(); plt.show()

---
## 1. The one unbreakable rule

To test a forecaster you must hold back data it never sees — but with time series there is exactly one legal way to do it:

> **Train on the past. Test on the future. Never shuffle.**

Randomly splitting time-series rows (standard practice elsewhere in statistics) lets the model train on March while being "tested" on January — it has seen the future. That is **look-ahead bias**, Module 1's first horseman, wearing a methodology costume. The split is a date, full stop.

In [ ]:
split = "2025-01-31"
train, test = m[:split], m[split:][1:]     # test starts AFTER the split month
print(f"Train: {len(train)} months (to {train.index[-1].date()}) | Test: {len(test)} months")

---
## 2. Three humble forecasters

Each predicts next month from information available *up to* this month — say it out loud each time; that sentence is the look-ahead vaccine.

**Naive**: tomorrow = today. The village idiot of forecasting — and the reigning champion on more series than any textbook admits.
**Moving average (k)**: tomorrow = mean of the last k months. Smooths noise, but drags its feet on trends.
**Simple exponential smoothing (SES)**: tomorrow = a weighted blend where recent months matter most; one dial, alpha, sets how fast old data fades. We implement it in five lines — no library, full understanding:

In [ ]:
def ses_forecasts(series, alpha):
    """Level after each observation; forecast for t+1 is the level at t."""
    level = series.iloc[0]
    fcs = []
    for actual in series:
        fcs.append(level)                      # forecast made BEFORE seeing 'actual'
        level = alpha*actual + (1-alpha)*level # then update with what happened
    return pd.Series(fcs, index=series.index)

# One-step-ahead forecasts over the WHOLE series (each uses only the past - check the loop order!)
f_naive = m.shift(1)
f_ma3   = m.shift(1).rolling(3).mean()
f_ses   = ses_forecasts(m, alpha=0.5)

fc = pd.DataFrame({"actual": m, "naive": f_naive, "ma3": f_ma3, "ses": f_ses}).dropna()
fc.tail(4).round(0)

*Pause on the SES loop:* the forecast is appended **before** the level updates. Swap those two lines and every forecast quietly contains the month it's predicting — a one-line look-ahead bug of the kind that has "validated" real trading systems. The order of two lines. That's how thin the ice is.

---
## 3. Scoring — three rulers, three personalities

| Metric | Formula in words | Personality |
|---|---|---|
| **MAE** | average of \|error\| | Democratic: every miss counts its size |
| **RMSE** | root of average squared error | Punishes big misses hard - use when disasters cost disproportionately |
| **MAPE** | average of \|error\|/actual | Speaks percent - comparable across series, but explodes near zero values |

In [ ]:
test_fc = fc.loc[test.index]

def score(pred, actual):
    e = actual - pred
    return pd.Series({"MAE": e.abs().mean(),
                      "RMSE": np.sqrt((e**2).mean()),
                      "MAPE%": (e.abs()/actual).mean()*100})

board = pd.DataFrame({name: score(test_fc[name], test_fc["actual"])
                      for name in ["naive","ma3","ses"]}).T.round(1)
print("TEST-PERIOD LEADERBOARD (2025):")
print(board.sort_values("RMSE"))

**Read the leaderboard slowly.** The naive forecast — *tomorrow equals today* — is at or near the top. The moving average, which sounds more sophisticated, typically trails it (averaging drags behind a trending series). This isn't an accident of our data; it's the shape of things to come in section 5.

The rule this leaderboard enforces forever:

> **No forecast has proven anything until it beats naive on honest, out-of-sample data.** Naive is the floor. A model that loses to "tomorrow = today" is not a model; it's overhead.

### ✏️ Exercise 1
Tune alpha: score SES for alpha in [0.2, 0.4, 0.6, 0.8] **on the test set**. Then the trick question: having picked the best alpha *using the test set*, is your reported score still honest? (No — you just used the test set to choose, which is a mild form of the same sin. The clean protocol: pick alpha on train, report on test, touch test once.)

In [ ]:
# your code here


---
## 4. Walk-forward: how evaluation is really done

One split is one exam. **Walk-forward** re-sits it every month: stand at month t, forecast t+1 using only data through t, step forward, repeat. Every forecast is out-of-sample; the result is a *track record*, not a single grade.

In [ ]:
horizon = 24     # walk the last 24 months
records = []
for i in range(len(m)-horizon, len(m)):
    history = m.iloc[:i]                    # ONLY the past
    actual  = m.iloc[i]
    records.append({
        "date": m.index[i], "actual": actual,
        "naive": history.iloc[-1],
        "ma3":   history.iloc[-3:].mean(),
        # SES forecast for the NEXT month = level updated through the last observation:
        "ses":   0.5*history.iloc[-1] + 0.5*ses_forecasts(history, 0.5).iloc[-1],
    })
wf = pd.DataFrame(records).set_index("date")

print("WALK-FORWARD LEADERBOARD (last 24 months):")
print(pd.DataFrame({c: score(wf[c], wf["actual"]) for c in ["naive","ma3","ses"]}).T.round(1).sort_values("RMSE"))

In [ ]:
# Watch the errors through time - is any model consistently better, or just lucky in patches?
err = wf[["naive","ma3","ses"]].sub(wf["actual"], axis=0).abs()
err.rolling(6).mean().plot(figsize=(9,3.2), title="6-month rolling MAE - the honest movie, not the single snapshot")
plt.ylabel("abs error"); plt.tight_layout(); plt.show()

### ✏️ Exercise 2
Extend the walk-forward with a fourth forecaster: **drift** — naive plus the average historical monthly change (`history.iloc[-1] + history.diff().mean()`). Drift knows the series trends up. Does that knowledge beat plain naive over the 24 months — and by enough to matter?

---
## 5. The punchline about markets

Why is naive so hard to beat *on prices*? Because if next month's move were reliably predictable from the past, traders would trade it **today**, moving today's price until the predictability vanished. Prices eat their own forecasts. The residue is a series close to a **random walk** — where "tomorrow = today" is, by construction, near-optimal.

Three consequences worth carrying out of this module:

1. **Beating naive on asset prices is extraordinarily hard**, and any backtest that does it easily should be assumed broken (look-ahead, survivorship, costs ignored) until proven otherwise. Module 11 inherits this assumption as its opening stance.
2. **Not everything is a price.** Revenue, transaction volumes, loan losses, electricity demand — series with real seasonality and inertia — are genuinely forecastable, and that's where these humble tools earn salaries. (Module 9 forecasts exactly such series.)
3. **The evaluation discipline transfers even where forecastability doesn't.** Time-respecting splits, naive floors, walk-forward, intervals: that harness is the module. The models are almost incidental.

---
## The ML border — a signpost, as promised

Everything here had a formula you could read: a line, an average, a five-line loop. One step further live regularised regressions, tree ensembles, gradient boosting, neural networks — models that *learn* their structure from data. They are powerful, genuinely useful, and **out of scope by design**: they belong to the ML course, where they get the room they deserve.

What you take across that border is worth more than any model: **the harness.** Every ML forecaster, however exotic, is judged by exactly the machinery you just built — honest splits, naive baselines, walk-forward, intervals, and the four biases standing guard. People who learn ML without the harness produce impressive numbers that evaporate in production. You will not be those people.

---
*AI disclosure: ______*

In [ ]:
# workspace
